In [76]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [77]:
print(torch.__version__)
print("GPU available: ", torch.cuda.is_available())

2.14.0+cpu
GPU available:  False


In [78]:
with open("training_data.txt", "r", encoding="utf-8") as f :
    text = f.read()

print('characters : ', len(text))
print()
print(text[:500])

characters :  1420167

BOOK I.
The quarrel between Agamemnon and Achilles—Achilles withdraws from the war, and sends his mother Thetis to ask Jove to help the Trojans—Scene between Jove and Juno on Olympus.

Sing, O goddess, the anger of Achilles son of Peleus, that brought countless ills upon the Achaeans. Many a brave soul did it send hurrying down to Hades, and many a hero did it yield a prey to dogs and vultures, for so were the counsels of Jove fulfilled from the day on which the son of Atreus, king of men, and g


In [79]:
chars = sorted(list(set(text)))

vocab_size = len(chars)
print('Vocabulary size : ', vocab_size)

Vocabulary size :  80


In [80]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

In [81]:
def encode(s):
    return [stoi[c] for c in s]

def decode(numbers) :
    return "".join(itos[i] for i in numbers)

In [82]:
message = "sing to me of the man"
encoded = encode(message)
print(encoded)
print(decode(encoded))

[67, 57, 62, 55, 1, 68, 63, 1, 61, 53, 1, 63, 54, 1, 68, 56, 53, 1, 61, 49, 62]
sing to me of the man


In [83]:
data = torch.tensor(encode(text), dtype = torch.long)

print(data.shape)
print(data.dtype)
print(data[:50])

torch.Size([1420167])
torch.int64
tensor([22, 35, 35, 31,  1, 29,  7,  0, 40, 56, 53,  1, 65, 69, 49, 66, 66, 53,
        60,  1, 50, 53, 68, 71, 53, 53, 62,  1, 21, 55, 49, 61, 53, 61, 62, 63,
        62,  1, 49, 62, 52,  1, 21, 51, 56, 57, 60, 60, 53, 67])


In [84]:
n = int(.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print('training tokens : ', len(train_data))
print('validation tokens : ', len(val_data))

training tokens :  1278150
validation tokens :  142017


In [85]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]

print('x :', x)
print('y : ', y)

print('x decoded : ', repr(decode(x.tolist())))
print('y decoded : ', repr(decode(y.tolist())))

x : tensor([22, 35, 35, 31,  1, 29,  7,  0])
y :  tensor([35, 35, 31,  1, 29,  7,  0, 40])
x decoded :  'BOOK I.\n'
y decoded :  'OOK I.\nT'


In [86]:
for t in range(block_size):
    context = x[:t +1]
    target = y[t]

    print(
        f"When input is {repr(decode(context.tolist()))},"
        f"target is {repr(decode([target.item()]))}"
    )

When input is 'B',target is 'O'
When input is 'BO',target is 'O'
When input is 'BOO',target is 'K'
When input is 'BOOK',target is ' '
When input is 'BOOK ',target is 'I'
When input is 'BOOK I',target is '.'
When input is 'BOOK I.',target is '\n'
When input is 'BOOK I.\n',target is 'T'


In [87]:
torch.manual_seed(42)

batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x, y

In [88]:
xb, yb = get_batch("train")

print("Inputs:")
print(xb)

print("\nTargets:")
print(yb)

print("\nShape:", xb.shape)

Inputs:
tensor([[63, 54,  1, 21, 62, 51, 56, 57],
        [68, 69, 61,  7, 47, 14, 48,  1],
        [52, 63,  1, 49, 60, 60,  5,  1],
        [ 1, 68, 56, 53,  1, 64, 63, 57]])

Targets:
tensor([[54,  1, 21, 62, 51, 56, 57, 67],
        [69, 61,  7, 47, 14, 48,  1, 35],
        [63,  1, 49, 60, 60,  5,  1, 63],
        [68, 56, 53,  1, 64, 63, 57, 62]])

Shape: torch.Size([4, 8])


In [89]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss
        
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
        
            logits, loss = self(idx)

      
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

     
            idx_next = torch.multinomial(probs, num_samples=1)

       
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [90]:
model = BigramLanguageModel(vocab_size)

logits, loss = model(xb, yb)

print("Logits Shape:", logits.shape)
print("Loss:", loss.item())

Logits Shape: torch.Size([32, 80])
Loss: 4.990232467651367


In [91]:
print(
    "Number of parameters:",
    sum(p.numel() for p in model.parameters())
)

Number of parameters: 6400


In [92]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = .001
)

In [93]:
xb, yb = get_batch("train")


logits, loss = model(xb, yb)


print("Loss befor update : ", loss.item())

optimizer.zero_grad(set_to_none=True)


loss.backward()


optimizer.step()


logits, loss_after = model(xb,yb)

print("Loss after update : ", loss_after.item())

Loss befor update :  4.994661808013916
Loss after update :  4.992709159851074


In [94]:
max_steps = 1000

for step in range(max_steps):
    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}: loss = {loss.item():.4f}")

Step 0: loss = 4.5177
Step 100: loss = 4.7254
Step 200: loss = 4.5901
Step 300: loss = 4.7166
Step 400: loss = 4.2153
Step 500: loss = 4.5044
Step 600: loss = 4.1717
Step 700: loss = 4.4189
Step 800: loss = 4.0591
Step 900: loss = 4.3178


In [95]:
start = torch.zeros((1, 1), dtype=torch.long)

generated = model.generate(start, max_new_tokens=500)

In [96]:
generated

tensor([[ 0, 45, 46, 15, 31, 40, 67, 55, 70, 77, 67, 21, 76,  9, 75, 75, 54, 16,
         60, 69, 38,  3, 41, 23, 58, 31, 76, 58,  3,  9, 35, 79, 29, 37, 21, 31,
         42,  9, 78, 59, 76, 23, 35, 76, 41, 60, 40,  8, 36, 15,  3, 25, 57, 12,
         22, 61, 30, 67, 63, 76, 41, 69,  1, 25, 23, 54,  8, 36, 73, 28, 73,  4,
         36, 52, 66, 19, 58, 13, 58,  3, 65, 16, 57, 39, 45, 76, 70, 30, 64, 74,
          2, 67,  1, 15, 53, 21, 53, 21, 77, 34, 41, 73,  4, 21, 68, 38, 74, 49,
         48, 47,  4, 14, 15, 42,  9,  3, 46, 38, 16, 72, 63, 51, 19, 14,  8,  7,
         56, 14, 76, 23, 76, 13, 72,  6, 23, 76, 52, 61, 35, 38, 65, 13,  5, 39,
         21, 32, 70, 55, 76, 57, 54, 11, 19, 78, 59, 24,  1, 36, 73, 67, 32, 77,
         22, 33, 29, 11, 54, 68, 66, 54, 28, 13, 68,  6,  5, 58, 35, 13, 14, 13,
         37, 74, 49,  8, 39, 76, 55, 56, 46, 15, 17, 76, 61,  9, 57, 18, 37, 34,
         64, 39, 13, 50, 63, 46, 14, 11, 59, 15,  4, 63, 41, 75, 51,  8, 57, 20,
         37, 50, 49, 72, 38,

In [97]:
print(decode(generated[0].tolist()))


YZ7KTsgv’sA‘1——f8luR(UCjK‘j(1O”IQAKV1“k‘CO‘UlT0P7(Ei4BmJso‘Uu ECf0PyHy)Pdr;j5j(q8iSY‘vJpz!s 7eAeA’NUy)AtRza][)67V1(ZR8xoc;60.h6‘C‘5x-C‘dmORq5,SALvg‘if3;“kD PysL’BMI3ftrfH5t-,jO565Qza0S‘ghZ79‘m1i:QNpS5boZ63k7)oU—c0i?QbaxR‘]ApWFgpf5”(J‘[WBc,33Hdrrfp6k.’BAjUfiqFECf 6TJ!)?[ONK2lvo7SZxCo7KoUV9zgN.(5”Q1KA athpza0iShA9F—]qcmm
dk AyZHcadJK’”mxGi:2!kDFR6LZE1OGE6!V4or,3ZO6”D,“Z(50is b)cy.vY8;E?He?[J9DeOJ]EP:9SJ’Jon—Y6j)elh”i!Rb8Ir7xR”banf.boNvvXQi?H5‘Urv
‘:NNK[-K‘ZR;5jm3rzoc,pk!D-JfC‘kbuNDoE(4!ata0w cH[Qx


In [98]:
print("Vocabulary size:", vocab_size)
print("Vocabulary characters:")
print(repr("".join(chars)))

Vocabulary size: 80
Vocabulary characters:
'\n !(),-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz—‘’“”'


In [99]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}

    for split in ["train", "val"]:
        losses = torch.zeros(100)

        for k in range(100):
            xb, yb = get_batch(split)
            logits, loss = model(xb, yb)
            losses[k] = loss.item()

        results[split] = losses.mean().item()

    model.train()
    return results

print(estimate_loss())

{'train': 4.023319244384766, 'val': 4.007937908172607}


In [100]:
print(model)
print("Number of parameters:", sum(
    p.numel() for p in model.parameters()
))

BigramLanguageModel(
  (token_embedding_table): Embedding(80, 80)
)
Number of parameters: 6400


In [101]:
class ContextLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size, embedding_dim=32):
        super().__init__()

        self.block_size = block_size

        self.token_embedding_table = nn.Embedding(
            vocab_size, embedding_dim
        )

     
        self.linear = nn.Linear(
            block_size * embedding_dim, vocab_size
        )

    def forward(self, idx, targets=None):
      
        embeddings = self.token_embedding_table(idx)

        B, T, C = embeddings.shape
        embeddings = embeddings.view(B, T * C)

    
        logits = self.linear(embeddings)

        if targets is None:
            loss = None
        else:
        
            targets = targets[:, -1]
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
   
            idx_cond = idx[:, -self.block_size:]

            logits, loss = self(idx_cond)

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [102]:
context_model = ContextLanguageModel(
    vocab_size=vocab_size,
    block_size=block_size
)

print(context_model)

print(
    "Number of parameters:",
    sum(p.numel() for p in context_model.parameters())
)

ContextLanguageModel(
  (token_embedding_table): Embedding(80, 32)
  (linear): Linear(in_features=256, out_features=80, bias=True)
)
Number of parameters: 23120


In [103]:
xb, yb = get_batch("train")


logits, loss = context_model(xb, yb)

print("Input shape:", xb.shape)
print("Target batch shape:", yb.shape)
print("Logits shape:", logits.shape)
print("Loss:", loss.item())

Input shape: torch.Size([4, 8])
Target batch shape: torch.Size([4, 8])
Logits shape: torch.Size([4, 80])
Loss: 4.913265228271484


In [104]:

context_optimizer = torch.optim.AdamW(
    context_model.parameters(),
    lr=0.001
)

max_steps = 1000

for step in range(max_steps):
    xb, yb = get_batch("train")

    logits, loss = context_model(xb, yb)

    context_optimizer.zero_grad(set_to_none=True)

    loss.backward()

    context_optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}: loss = {loss.item():.4f}")

Step 0: loss = 4.1693
Step 100: loss = 2.9150
Step 200: loss = 3.6718
Step 300: loss = 2.7628
Step 400: loss = 3.2116
Step 500: loss = 2.1289
Step 600: loss = 3.2830
Step 700: loss = 2.0278
Step 800: loss = 1.9358
Step 900: loss = 1.9134


In [105]:
@torch.no_grad()
def estimate_context_loss():
    context_model.eval()
    results = {}

    for split in ["train", "val"]:
        losses = torch.zeros(100)

        for k in range(100):
            xb, yb = get_batch(split)
            logits, loss = context_model(xb, yb)
            losses[k] = loss.item()

        results[split] = losses.mean().item()

    context_model.train()
    return results

print(estimate_context_loss())

{'train': 2.575263023376465, 'val': 2.4579861164093018}


In [106]:
start = torch.zeros((1, block_size), dtype=torch.long)

generated = context_model.generate(
    start,
    max_new_tokens=500
)

print(decode(generated[0].tolist()))









hTt gnantmathenP “hor ialieew shilo thr  oom tnszo; nroolund thee aasebtse T;nnglrgdVd, lyenet st tutlrmgIy his tars, old tut toly phasro
 sro zhig bwear hho tht  oou ith he hi h te th iou iroil wedsboo jfand ‘oW fooig
 9kte ther th t oge slojdAdousOnI,y
wwcuEh“cT—Wh[oI.y.lPugm[o aitht fado o dee thiyhhsy e vere os iaedrofePhor ,ons antyh men foo tr,w trot, und inttof thraty ty,ht  eit lultoslfwan fshamagEtimf woe faie toepr s te mhet ca aliopt  hei aado strether swa  hereeave thbsls pt alyd ios


In [107]:
class ContextLanguageModelV2(nn.Module):
    def __init__(self, vocab_size, block_size, embedding_dim=32):
        super().__init__()

        self.block_size = block_size

        self.token_embedding_table = nn.Embedding(
            vocab_size, embedding_dim
        )

        self.linear = nn.Linear(
            block_size * embedding_dim, vocab_size
        )

    def forward(self, idx, targets=None):
        B, T = idx.shape

        embeddings = self.token_embedding_table(idx)

        embeddings = F.pad(
            embeddings,
            (0, 0, self.block_size - 1, 0)
        )

        contexts = embeddings.unfold(
            dimension=1,
            size=self.block_size,
            step=1
        )

        contexts = contexts.permute(0, 1, 3, 2)
        contexts = contexts.reshape(
            B, T, self.block_size * embeddings.shape[-1]
        )

        logits = self.linear(contexts)

        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(
                logits.reshape(B * T, -1),
                targets.reshape(B * T)
            )

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):

  
            idx_cond = idx[:, -self.block_size:]

            if idx_cond.shape[1] < self.block_size:
                padding = self.block_size - idx_cond.shape[1]
                idx_cond = F.pad(idx_cond, (padding, 0))


            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)


            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [108]:
context_model_v2 = ContextLanguageModelV2(
    vocab_size=vocab_size,
    block_size=block_size
)

xb, yb = get_batch("train")

logits, loss = context_model_v2(xb, yb)

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)
print("Logits shape:", logits.shape)
print("Loss:", loss.item())

Input shape: torch.Size([4, 8])
Target shape: torch.Size([4, 8])
Logits shape: torch.Size([4, 8, 80])
Loss: 4.474668502807617


In [109]:
optimizer_v2 = torch.optim.AdamW(
    context_model_v2.parameters(),
    lr=0.001
)

max_steps = 1000

for step in range(max_steps):

    xb, yb = get_batch("train")

    logits, loss = context_model_v2(xb, yb)

    optimizer_v2.zero_grad(set_to_none=True)

    loss.backward()

    optimizer_v2.step()

    if step % 100 == 0:
        print(f"Step {step}: loss = {loss.item():.4f}")

Step 0: loss = 4.5167
Step 100: loss = 3.0075
Step 200: loss = 2.9994
Step 300: loss = 2.6556
Step 400: loss = 2.4971
Step 500: loss = 3.1078
Step 600: loss = 2.5503
Step 700: loss = 3.3406
Step 800: loss = 2.0792
Step 900: loss = 2.5567


In [110]:
@torch.no_grad()
def estimate_v2_loss():
    context_model_v2.eval()
    results = {}

    for split in ["train", "val"]:
        losses = torch.zeros(100)

        for k in range(100):
            xb, yb = get_batch(split)

            logits, loss = context_model_v2(xb, yb)

            losses[k] = loss.item()

        results[split] = losses.mean().item()

    context_model_v2.train()
    return results

print(estimate_v2_loss())

{'train': 2.3400774002075195, 'val': 2.3622379302978516}


In [111]:

start = torch.zeros((1, 1), dtype=torch.long)

generated = context_model_v2.generate(
    start,
    max_new_tokens=1000
)

print(decode(generated[0].tolist()))


i,ra,enilisg and fo thet yousesthe dusilly tang ct rirucoutt he Thevd wtoty sid s th thitt hecharadd . ver dor tog theaedt yhe rou deit, deld,e f whing to th Td sher inls th he we fto her ind wh wirheand wian the hillls, f olofs.s anr wire dove f mt list or mit tia sh uldhe nrus he Arikare to itrelygoul Th thy t orlyoucrey boulond wpspd tust hem lavts m hethey fFomedyos ay Ars apuenbthe norrtie thle niyrher he ond  ap arlyon mtor tir mom ife mat, furtowisyeurd optuthany thearpmobs bothilck

“ong, fniy mad the .r. ay mald in, bnppofkrt hes chor doeas wrer our noe ie mil. wivcrons hovr, gye and dais hart hyingyt whe bond rok limfnd Wu.ourar adon, T but usot Hfon hedrth
“roo wired ham nwelc while, “heabte ther yitveren. not Ao Himenm art thengit hadelisswat as Shave notr mgly lante hor whinTcme toredfon, Bucbntns handw,pbind yse art wy ild gime th he btearusded etvespir ffrthom yndand heake t tisponw so lo thimg the wile himd mingurued dflling ta eed de wamndethof toutmes, .jd mamose lle

In [112]:
class SelfAttentionHead(nn.Module):
    def __init__(self, embedding_dim, head_size, block_size):
        super().__init__()

        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape

        keys = self.key(x)
        queries = self.query(x)

        scores = queries @ keys.transpose(-2, -1)
        scores = scores * (keys.shape[-1] ** -0.5)

        scores = scores.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        weights = F.softmax(scores, dim=-1)

        values = self.value(x)

        output = weights @ values

        return output, weights

In [113]:
head = SelfAttentionHead(
    embedding_dim=32,
    head_size=16,
    block_size=8
)

x = torch.randn(1, 8, 32)

output, weights = head(x)

print("Output shape:", output.shape)
print("Attention weights shape:", weights.shape)

print("\nAttention weights:")
print(weights[0])

Output shape: torch.Size([1, 8, 16])
Attention weights shape: torch.Size([1, 8, 8])

Attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4646, 0.5354, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3390, 0.2492, 0.4118, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2559, 0.2457, 0.2257, 0.2727, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1903, 0.2102, 0.1750, 0.1975, 0.2270, 0.0000, 0.0000, 0.0000],
        [0.2029, 0.1095, 0.2034, 0.1728, 0.1547, 0.1568, 0.0000, 0.0000],
        [0.1021, 0.0772, 0.1929, 0.1059, 0.1642, 0.2520, 0.1058, 0.0000],
        [0.0884, 0.1051, 0.2334, 0.0892, 0.1005, 0.0917, 0.1370, 0.1546]],
       grad_fn=<SelectBackward0>)


In [114]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, embedding_dim, block_size):
        super().__init__()

        head_size = embedding_dim // num_heads

        self.heads = nn.ModuleList([
            SelfAttentionHead(
                embedding_dim,
                head_size,
                block_size
            )
            for _ in range(num_heads)
        ])

        self.projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )

    def forward(self, x):
        outputs = [
            head(x)[0] for head in self.heads
        ]

        combined = torch.cat(outputs, dim=-1)

        return self.projection(combined)

In [115]:
multi_head = MultiHeadAttention(
    num_heads=4,
    embedding_dim=32,
    block_size=8
)


x = torch.randn(4, 8, 32)

output = multi_head(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([4, 8, 32])
Output shape: torch.Size([4, 8, 32])


In [116]:
parameters = sum(
    p.numel() for p in multi_head.parameters()
)

print("Multi-head attention parameters:", parameters)

Multi-head attention parameters: 4128


In [117]:
class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim)
        )

    def forward(self, x):
        return self.network(x)

In [118]:
feed_forward = FeedForward(embedding_dim=32)

x = torch.randn(4, 8, 32)

output = feed_forward(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

parameters = sum(
    p.numel() for p in feed_forward.parameters()
)

print("Feed-forward parameters:", parameters)

Input shape: torch.Size([4, 8, 32])
Output shape: torch.Size([4, 8, 32])
Feed-forward parameters: 8352


In [119]:
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, block_size):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads,
            embedding_dim,
            block_size
        )

        self.feed_forward = FeedForward(embedding_dim)

        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):

      
        x = x + self.attention(self.ln1(x))

       
        x = x + self.feed_forward(self.ln2(x))

        return x

In [120]:
transformer_block = TransformerBlock(
    embedding_dim=32,
    num_heads=4,
    block_size=8
)

x = torch.randn(4, 8, 32)

output = transformer_block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

parameters = sum(
    p.numel() for p in transformer_block.parameters()
)

print("Transformer block parameters:", parameters)

Input shape: torch.Size([4, 8, 32])
Output shape: torch.Size([4, 8, 32])
Transformer block parameters: 12608


In [121]:
class GPTLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        block_size,
        num_heads,
        num_layers
    ):
        super().__init__()

        self.block_size = block_size

   
        self.token_embedding_table = nn.Embedding(
            vocab_size, embedding_dim
        )

     
        self.position_embedding_table = nn.Embedding(
            block_size, embedding_dim
        )

        self.blocks = nn.Sequential(*[
            TransformerBlock(
                embedding_dim,
                num_heads,
                block_size
            )
            for _ in range(num_layers)
        ])

    
        self.ln_final = nn.LayerNorm(embedding_dim)

   
        self.lm_head = nn.Linear(
            embedding_dim, vocab_size
        )

    def forward(self, idx, targets=None):
        B, T = idx.shape

   
        token_embeddings = self.token_embedding_table(idx)

   
        positions = torch.arange(T, device=idx.device)
        position_embeddings = self.position_embedding_table(positions)

  
        x = token_embeddings + position_embeddings

     
        x = self.blocks(x)

      
        x = self.ln_final(x)

        
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(
                logits.reshape(B * T, -1),
                targets.reshape(B * T)
            )

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):

           
            idx_cond = idx[:, -self.block_size:]

           
            logits, loss = self(idx_cond)

           
            logits = logits[:, -1, :]

           
            probs = F.softmax(logits, dim=-1)

            
            idx_next = torch.multinomial(probs, num_samples=1)

            
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


In [122]:
gpt_model = GPTLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=32,
    block_size=8,
    num_heads=4,
    num_layers=2
)

In [123]:
parameters = sum(
    p.numel() for p in gpt_model.parameters()
)

print("Total GPT parameters:", parameters)

Total GPT parameters: 30736


In [124]:
xb, yb = get_batch("train")

logits, loss = gpt_model(xb, yb)

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)
print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

Input shape: torch.Size([4, 8])
Target shape: torch.Size([4, 8])
Logits shape: torch.Size([4, 8, 80])
Initial loss: 4.650191783905029


In [125]:
optimizer_gpt = torch.optim.AdamW(
    gpt_model.parameters(),
    lr=0.001
)

In [126]:
max_steps = 2000

for step in range(max_steps):

   
    xb, yb = get_batch("train")

   
    logits, loss = gpt_model(xb, yb)

   
    optimizer_gpt.zero_grad(set_to_none=True)

    
    loss.backward()

   
    optimizer_gpt.step()

   
    if step % 100 == 0:
        print(f"Step {step}: loss = {loss.item():.4f}")

Step 0: loss = 4.7166
Step 100: loss = 2.7526
Step 200: loss = 2.5987
Step 300: loss = 2.8309
Step 400: loss = 2.0964
Step 500: loss = 2.4650
Step 600: loss = 2.5113
Step 700: loss = 2.5383
Step 800: loss = 2.8032
Step 900: loss = 2.1704
Step 1000: loss = 2.3148
Step 1100: loss = 2.1443
Step 1200: loss = 2.4635
Step 1300: loss = 2.3500
Step 1400: loss = 2.2013
Step 1500: loss = 2.1358
Step 1600: loss = 2.3161
Step 1700: loss = 2.4075
Step 1800: loss = 2.3388
Step 1900: loss = 2.1915


In [127]:
@torch.no_grad()
def estimate_gpt_loss():
    gpt_model.eval()

    results = {}

    for split in ["train", "val"]:
        losses = torch.zeros(100)

        for k in range(100):
            xb, yb = get_batch(split)
            logits, loss = gpt_model(xb, yb)
            losses[k] = loss.item()

        results[split] = losses.mean().item()

    gpt_model.train()

    return results

In [128]:
gpt_results = estimate_gpt_loss()

print(gpt_results)

{'train': 2.2378320693969727, 'val': 2.2983484268188477}


In [129]:
gpt_weights = gpt_model.state_dict()

In [130]:
gpt_model = GPTLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=32,
    block_size=8,
    num_heads=4,
    num_layers=2
)

gpt_model.load_state_dict(gpt_weights)

<All keys matched successfully>

In [131]:
prompt = "Achilles"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
)

generated = gpt_model.generate(
    context,
    max_new_tokens=500
)

print(decode(generated[0].tolist()))

Achilles thun chert yhind malde whe shich8]s thasto his as Cenl wibCpleng thoa then Sthe To to ering stay che thly soo this bloughtirt istht prurising, thalIs goncut ohe, Mor s thayy
ily th ith the llyseato lof Anos se tint cang sus th”
Jaill seat thies ing nofenind ght the, man wingitm ullod; fy fourd cuy,, wis, Enth “Mich iv
Th
“Ackecwang. bly
Hrint courntiyty

Alle, ye cighsn shewn swiss yium nond Adalll ha thive trik cheng feing mighd son ome to tho tho chus a es goos hald witheance th sulllodseenti


In [132]:
print("Context shape:", context.shape)
print("Generated shape:", generated.shape)
print("Generated tokens:", generated.tolist())

Context shape: torch.Size([1, 8])
Generated shape: torch.Size([1, 508])
Generated tokens: [[21, 51, 56, 57, 60, 60, 53, 67, 1, 68, 56, 69, 62, 1, 51, 56, 53, 66, 68, 1, 73, 56, 57, 62, 52, 1, 61, 49, 60, 52, 53, 1, 71, 56, 53, 1, 67, 56, 57, 51, 56, 16, 48, 67, 1, 68, 56, 49, 67, 68, 63, 1, 56, 57, 67, 1, 49, 67, 1, 23, 53, 62, 60, 1, 71, 57, 50, 23, 64, 60, 53, 62, 55, 1, 68, 56, 63, 49, 1, 68, 56, 53, 62, 1, 39, 68, 56, 53, 1, 40, 63, 1, 68, 63, 1, 53, 66, 57, 62, 55, 1, 67, 68, 49, 73, 1, 51, 56, 53, 1, 68, 56, 60, 73, 1, 67, 63, 63, 1, 68, 56, 57, 67, 1, 50, 60, 63, 69, 55, 56, 68, 57, 66, 68, 1, 57, 67, 68, 56, 68, 1, 64, 66, 69, 66, 57, 67, 57, 62, 55, 5, 1, 68, 56, 49, 60, 29, 67, 1, 55, 63, 62, 51, 69, 68, 1, 63, 56, 53, 5, 1, 33, 63, 66, 1, 67, 1, 68, 56, 49, 73, 73, 0, 57, 60, 73, 1, 68, 56, 1, 57, 68, 56, 1, 68, 56, 53, 1, 60, 60, 73, 67, 53, 49, 68, 63, 1, 60, 63, 54, 1, 21, 62, 63, 67, 1, 67, 53, 1, 68, 57, 62, 68, 1, 51, 49, 62, 55, 1, 67, 69, 67, 1, 68, 56, 79, 0, 30, 49

In [133]:
import inspect

print(inspect.getsource(gpt_model.generate))

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):


            idx_cond = idx[:, -self.block_size:]


            logits, loss = self(idx_cond)


            logits = logits[:, -1, :]


            probs = F.softmax(logits, dim=-1)


            idx_next = torch.multinomial(probs, num_samples=1)


            idx = torch.cat((idx, idx_next), dim=1)

        return idx



In [134]:
block_size = 64

In [135]:
gpt_model_v2 = GPTLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=32,
    block_size=64,
    num_heads=4,
    num_layers=2
)

print(
    "Parameters:",
    sum(p.numel() for p in gpt_model_v2.parameters())
)

Parameters: 32528


In [136]:
xb, yb = get_batch("train")

logits, loss = gpt_model_v2(xb, yb)

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)
print("Output shape:", logits.shape)
print("Initial loss:", loss.item())

Input shape: torch.Size([4, 64])
Target shape: torch.Size([4, 64])
Output shape: torch.Size([4, 64, 80])
Initial loss: 4.664641857147217


In [137]:
optimizer_gpt_v2 = torch.optim.AdamW(
    gpt_model_v2.parameters(),
    lr=0.001
)

In [138]:
max_steps = 5000

for step in range(max_steps):
    xb, yb = get_batch("train")

    logits, loss = gpt_model_v2(xb, yb)

    optimizer_gpt_v2.zero_grad(set_to_none=True)

    loss.backward()

    optimizer_gpt_v2.step()

    if step % 250 == 0:
        print(f"Step {step}: loss = {loss.item():.4f}")

Step 0: loss = 4.6804
Step 250: loss = 2.6607
Step 500: loss = 2.3137
Step 750: loss = 2.3239
Step 1000: loss = 2.1464
Step 1250: loss = 2.2110
Step 1500: loss = 2.2110
Step 1750: loss = 2.1872
Step 2000: loss = 2.3408
Step 2250: loss = 2.2485
Step 2500: loss = 2.0315
Step 2750: loss = 2.1343
Step 3000: loss = 2.2459
Step 3250: loss = 2.2370
Step 3500: loss = 2.1281
Step 3750: loss = 2.1900
Step 4000: loss = 2.0866
Step 4250: loss = 1.9951
Step 4500: loss = 2.0918
Step 4750: loss = 1.9869


In [139]:
@torch.no_grad()
def estimate_gpt_v2_loss():
    gpt_model_v2.eval()

    results = {}

    for split in ["train", "val"]:
        losses = torch.zeros(100)

        for k in range(100):
            xb, yb = get_batch(split)

            logits, loss = gpt_model_v2(xb, yb)

            losses[k] = loss.item()

        results[split] = losses.mean().item()

    gpt_model_v2.train()

    return results

In [140]:
gpt_v2_results = estimate_gpt_v2_loss()
print(gpt_v2_results)

{'train': 2.057342767715454, 'val': 2.060382604598999}


In [141]:
prompt = "Achilles"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
)

generated_v2 = gpt_model_v2.generate(
    context,
    max_new_tokens=1000
)

print(decode(generated_v2[0].tolist()))

Achilles for tigh, ar ford she s cond, wo my you au day cowerse cos and aend con shieand coce whon. The soclve as sop now As and the the my sat re toa chis youd hehe sous whor shin dord theys a far hor are hear mis and the we crcomix ave the a gans dinot sen the bor, goutichfes, ind inglous aneoros in cande of tond onosen and I gouth hay you; for the shetenind If the warer dore nemin a a, Dup whe fe a a shep ghomede oveth s fieety Athe me wore is tha yone Pa. I “HiT torre for twe, thepas wedyden whin crys alling amfnrov, his and as Id pouivar sonk the woul as thous; of sain igs; afand no; an fard us foes dorleves. Bulf Hefin bes.

Salls sthe daeanchim fus kin hip so he fan whe y geand fi; bore buth lim sy as re wecus lanearing cus the ared minon bose shis sor kelot be gowes buges, benef gind I lidelling and and fleloer ond the shelly I Bone heimbuld wands this irls. Fy Ajagh burng ran fan iout baiint ve now ondstous. Dandele hor, and gut shin Tro hem haere wh sache-anmed And chey toor 

In [142]:
# BPE

In [143]:
tokens = list(text)

print(tokens[:100])
print("Total tokens:", len(tokens))

['B', 'O', 'O', 'K', ' ', 'I', '.', '\n', 'T', 'h', 'e', ' ', 'q', 'u', 'a', 'r', 'r', 'e', 'l', ' ', 'b', 'e', 't', 'w', 'e', 'e', 'n', ' ', 'A', 'g', 'a', 'm', 'e', 'm', 'n', 'o', 'n', ' ', 'a', 'n', 'd', ' ', 'A', 'c', 'h', 'i', 'l', 'l', 'e', 's', '—', 'A', 'c', 'h', 'i', 'l', 'l', 'e', 's', ' ', 'w', 'i', 't', 'h', 'd', 'r', 'a', 'w', 's', ' ', 'f', 'r', 'o', 'm', ' ', 't', 'h', 'e', ' ', 'w', 'a', 'r', ',', ' ', 'a', 'n', 'd', ' ', 's', 'e', 'n', 'd', 's', ' ', 'h', 'i', 's', ' ', 'm', 'o']
Total tokens: 1420167


In [144]:
def get_pair_counts(tokens):
    pair_counts = {}

    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])

        pair_counts[pair] = pair_counts.get(pair, 0) + 1

    return pair_counts

In [145]:
pair_counts = get_pair_counts(tokens)

print("Unique pairs:", len(pair_counts))
print("\nMost frequent pairs:")

for pair, count in sorted(
    pair_counts.items(),
    key=lambda item: item[1],
    reverse=True
)[:20]:
    print(pair, count)

Unique pairs: 1584

Most frequent pairs:
('e', ' ') 46874
(' ', 't') 39987
('h', 'e') 39685
('t', 'h') 36693
('d', ' ') 29786
(' ', 'a') 28476
('s', ' ') 26670
(' ', 'h') 23525
('t', ' ') 21936
('a', 'n') 21777
(' ', 's') 20153
(' ', 'w') 19524
(',', ' ') 19470
('n', ' ') 19164
(' ', 'o') 17144
('n', 'd') 16928
('e', 'r') 16672
('i', 'n') 16624
('r', ' ') 15735
('o', 'u') 15077


In [146]:
def merge_pair(tokens, pair, new_token):
    new_tokens = []
    i = 0

    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            new_tokens.append(new_token)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1

    return new_tokens

In [147]:
tokens = merge_pair(tokens, ('e', ' '), 'e ')

print("First 100 tokens after merge:")
print(tokens[:100])

print("\nTotal tokens after merge:", len(tokens))

First 100 tokens after merge:
['B', 'O', 'O', 'K', ' ', 'I', '.', '\n', 'T', 'h', 'e ', 'q', 'u', 'a', 'r', 'r', 'e', 'l', ' ', 'b', 'e', 't', 'w', 'e', 'e', 'n', ' ', 'A', 'g', 'a', 'm', 'e', 'm', 'n', 'o', 'n', ' ', 'a', 'n', 'd', ' ', 'A', 'c', 'h', 'i', 'l', 'l', 'e', 's', '—', 'A', 'c', 'h', 'i', 'l', 'l', 'e', 's', ' ', 'w', 'i', 't', 'h', 'd', 'r', 'a', 'w', 's', ' ', 'f', 'r', 'o', 'm', ' ', 't', 'h', 'e ', 'w', 'a', 'r', ',', ' ', 'a', 'n', 'd', ' ', 's', 'e', 'n', 'd', 's', ' ', 'h', 'i', 's', ' ', 'm', 'o', 't', 'h']

Total tokens after merge: 1373293


In [148]:
original_tokens = list(text)

print("Original token count:", len(original_tokens))
print("Current token count:", len(tokens))

Original token count: 1420167
Current token count: 1373293


In [149]:
tokens = list(original_tokens)

merges = {}

num_merges = 100

for i in range(num_merges):

    pair_counts = get_pair_counts(tokens)

    if not pair_counts:
        break

    best_pair = max(pair_counts, key=pair_counts.get)

    new_token = best_pair[0] + best_pair[1]

    tokens = merge_pair(tokens, best_pair, new_token)

    merges[best_pair] = new_token

    if (i + 1) % 10 == 0:
        print(f"Merge {i + 1}: {best_pair} -> {repr(new_token)}")

print("\nTotal merges:", len(merges))
print("Tokens remaining:", len(tokens))

Merge 10: ('th', 'e ') -> 'the '
Merge 20: ('l', 'l') -> 'll'
Merge 30: ('m', ' ') -> 'm '
Merge 40: ('e', 'n ') -> 'en '
Merge 50: ('hi', 's ') -> 'his '
Merge 60: ('I', ' ') -> 'I '
Merge 70: ('er', 'e ') -> 'ere '
Merge 80: ('r', 'a') -> 'ra'
Merge 90: ('hi', 'm ') -> 'him '
Merge 100: ('a', 't') -> 'at'

Total merges: 100
Tokens remaining: 769693


In [150]:
print("First 50 learned tokens:\n")

for i, (pair, new_token) in enumerate(merges.items()):
    print(f"{i + 1:3}: {repr(new_token)}")

    if i == 49:
        break

First 50 learned tokens:

  1: 'e '
  2: 'th'
  3: 'd '
  4: 's '
  5: 't '
  6: 'an'
  7: ', '
  8: 'n '
  9: 'er'
 10: 'the '
 11: 'ou'
 12: 'o '
 13: 'and '
 14: 'in'
 15: 'y '
 16: 'hi'
 17: 'f '
 18: 'or'
 19: 'of '
 20: 'll'
 21: 'ar'
 22: 'ed '
 23: 'to '
 24: 'ing'
 25: 'ha'
 26: 'the'
 27: 'er '
 28: '. '
 29: 'on'
 30: 'm '
 31: 'ea'
 32: 'll '
 33: 'en'
 34: 'he '
 35: 'ing '
 36: 'wi'
 37: 'you'
 38: ', and '
 39: 'es'
 40: 'en '
 41: 'on '
 42: 'ro'
 43: 'in '
 44: 'as '
 45: 'or '
 46: 'st'
 47: 'a '
 48: 'at '
 49: 'el'
 50: 'his '


In [151]:
vocab = list(chars)

for new_token in merges.values():
    if new_token not in vocab:
        vocab.append(new_token)

bpe_stoi = {token: i for i, token in enumerate(vocab)}
bpe_itos = {i: token for i, token in enumerate(vocab)}

bpe_vocab_size = len(vocab)

print("Original character vocabulary:", len(chars))
print("BPE vocabulary size:", bpe_vocab_size)

Original character vocabulary: 80
BPE vocabulary size: 180


In [152]:
def bpe_encode(text):
    tokens = list(text)

    for pair, new_token in merges.items():
        tokens = merge_pair(tokens, pair, new_token)

    return [bpe_stoi[token] for token in tokens]

In [153]:
def bpe_decode(token_ids):
    return "".join(bpe_itos[token_id] for token_id in token_ids)

In [154]:
sample_text = text[:500]

encoded = bpe_encode(sample_text)

decoded = bpe_decode(encoded)

print("Original text:")
print(sample_text[:200])

print("\nEncoded token IDs:")
print(encoded[:50])

print("\nDecoded text:")
print(decoded[:200])

print("\nDoes the decoded text match the original?")
print(decoded == sample_text)

Original text:
BOOK I.
The quarrel between Agamemnon and Achilles—Achilles withdraws from the war, and sends his mother Thetis to ask Jove to help the Trojans—Scene between Jove and Juno on Olympus.

Sing, O goddess

Encoded token IDs:
[22, 35, 35, 31, 1, 29, 7, 0, 40, 113, 65, 69, 100, 66, 128, 1, 162, 68, 71, 53, 119, 21, 55, 49, 61, 53, 61, 62, 120, 92, 21, 51, 95, 99, 118, 75, 21, 51, 95, 99, 137, 115, 81, 52, 159, 71, 83, 54, 121, 109]

Decoded text:
BOOK I.
The quarrel between Agamemnon and Achilles—Achilles withdraws from the war, and sends his mother Thetis to ask Jove to help the Trojans—Scene between Jove and Juno on Olympus.

Sing, O goddess

Does the decoded text match the original?
True
